In [ ]:
"""
 https://www.bambooweekly.com/earthquake/
 https://www.bambooweekly.com/earthquakes-solution/
 https://github.com/JoergEm/Bamboo-Weekly/tree/main 
"""

In [ ]:
# Imports
from IPython.display import FileLink, Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import seaborn as sns
display(Markdown("Imports ✅"))

In [ ]:
# SparkSession
spark = SparkSession.builder \
    .appName("Earthquakes") \
    .getOrCreate()

display(Markdown("SparkSession ✅"))

In [ ]:
# Function creating local folders
def create_folders(folders: list[str]) -> bool:
    try:
        for folder in folders:
            folderpath: str = os.path.join(os.getcwd(), folder)
            if not os.path.exists(folderpath):
                os.makedirs(folderpath, exist_ok=True)
    except:
        print("Folder {folderpath} could not be created.")
        return False
    else:
        display(Markdown("Folders ✅"))
        return True

In [ ]:
# Function downloading data locally
def download_data(url: str, filename: str) -> bool:
    from urllib.request import urlretrieve
    from urllib.error import HTTPError
    try:
        urlretrieve(url, filename)
        return True
    except HTTPError as e:
        if e.code == 403:
            import requests
            try:
                response: requests.Response = requests.get(url)
                with open(filename, 'wb') as f:
                    f.write(response.content)
                    return True
            except:
                print("Could not download Data")
                return False
    return False

In [ ]:
# Links and folders to recieve data and read into DataFrame
url: str  = 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2000-01-01&endtime=2000-12-31&minmagnitude=4.5'
filename: str  = 'query.csv'
folders: list[str] = ['data', 'results']
filepath: str  = os.path.join(folders[0], filename)
create_folders(folders)

if not os.path.exists(filepath):
    if download_data(url, filepath):
        data: pd.DataFrame = pd.read_csv(filepath, parse_dates=["time", "updated"], index_col="time")
        display(Markdown("Data ✅"))
    else:
        display(Markdown('Error ❌'))
else:
    data: pd.DataFrame = pd.read_csv(filepath, parse_dates=["time", "updated"], index_col="time")
    display(Markdown("Data loaded from existing file 📁")) 

if os.path.exists(filepath):
    display(FileLink(filepath))

data = data.reset_index() 
df = spark.createDataFrame(data)
display(Markdown("Data converted to Spark DataFrame ✅"))

In [ ]:
# Read the downloaded CSV file (which will be called `query.csv`, but which I renamed to `earthquake-data.csv` on my computer) into Pandas.
df.orderBy("time").select("time", "mag", "depth", "place", "type", "status").show(5, truncate=False)

In [ ]:
# How many seismic events take plac…
df.groupBy("type").count().show()